In [0]:
%pip install statsmodels

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
import itertools
import warnings

# 1. Data Loading & Formatting
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
df.index.freq = 'MS' # Set frequency to Monthly Start
print("shape:", df.shape)
print("index:", df.index)
print("columns:", df.columns)
print("dtypes:")
df.head()

Library Imports and Data Preparation:
 importing the necessary libraries for data manipulation, statistical testing, and time series modeling. 
 Use the classic "Airline Passengers" dataset, ensuring the Month column is parsed as a datetime object and set as the index.

In [0]:
# 2. Exploratory Data Analysis (EDA)
# Decomposing to see Trend and Seasonality
decomposition = seasonal_decompose(df['Passengers'], model='multiplicative')
decomposition.plot()
plt.show()

Exploratory Data Analysis (EDA): Seasonal Decomposition
Before modeling, decompose the series to visualize its underlying components: Trend, Seasonality, and Residuals (noise). Since the seasonal fluctuations increase in magnitude over time, we use a multiplicative model

In [0]:
# 3. Stationarity Check (Augmented Dickey-Fuller Test)
def check_stationarity(timeseries):
    result = adfuller(timeseries)
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    if result[1] <= 0.05:
        print("Data is stationary")
    else:
        print("Data is non-stationary (Differencing required)")

check_stationarity(df['Passengers'])

Testing for Stationarity SARIMAX models require the data to be stationary (constant mean and variance). Use the Augmented Dickey-Fuller (ADF) test. If the p-value is greater than 0.05, we fail to reject the null hypothesis, meaning the data is non-stationary and requires differencing ($d$).

In [0]:
# 4. SARIMAX Hyperparameter Grid Search (AIC Optimization)
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in list(itertools.product(p, d, q))]

warnings.filterwarnings("ignore")
best_aic = float("inf")
best_params = None

for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            temp_model = sm.tsa.statespace.SARIMAX(df['Passengers'],
                                                   order=param,
                                                   seasonal_order=param_seasonal,
                                                   enforce_stationarity=False,
                                                   enforce_invertibility=False)
            results = temp_model.fit(disp=False)
            if results.aic < best_aic:
                best_aic = results.aic
                best_params = (param, param_seasonal)
        except:
            continue

print(f'Best SARIMAX Model: {best_params[0]}x{best_params[1]} - AIC: {best_aic}')

Hyperparameter Optimization (Grid Search)A SARIMAX model is defined by $(p, d, q) \times (P, D, Q, s)$. To find the best combination, perform a grid search that minimizes the Akaike Information Criterion (AIC), which balances model fit with complexity.

In [0]:
# 5. Final Model Fitting & Diagnostics
model = sm.tsa.statespace.SARIMAX(df['Passengers'],
                                  order=best_params[0],
                                  seasonal_order=best_params[1])
results = model.fit()
results.plot_diagnostics(figsize=(12, 8))
plt.show()

# 6. Forecasting the next 24 Months
forecast = results.get_forecast(steps=24)
forecast_df = forecast.summary_frame()

# Plotting Results
plt.figure(figsize=(10, 5))
plt.plot(df['Passengers'], label='Historical Data')
plt.plot(forecast_df['mean'], label='Forecast', color='red')
plt.fill_between(forecast_df.index, forecast_df['mean_ci_lower'], forecast_df['mean_ci_upper'], color='pink', alpha=0.3)
plt.title('Airline Passenger Forecast (SARIMAX)')
plt.legend()
plt.show()

Model Fitting and Diagnostics
Using the optimized parameters, we fit the final model. Review the diagnostic plots to ensure the residuals are normally distributed and uncorrelated (white noise)